In [1]:
import pandas as pd

import plotly.graph_objects as go
import numpy as np

# 1. Lấy danh sách các quốc gia duy nhất (sắp xếp theo Alphabet)
countries = ['VNM', 'THA', 'PHL', 'SGP', 'CHN', 'IND', 'IRL', 'DEU', 'ZAF', 'USA', 'JPN']
countries.sort()

file_path = 'cleaned_data/information/panel_macro_cleaned.csv'
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Không tìm thấy file tại {file_path}.")
    
fig = go.Figure()

# 2. Vòng lặp: Thêm toàn bộ các traces (đường/cột) cho TẤT CẢ quốc gia vào biểu đồ
for country in countries:
    country_df = df[df['Country'] == country].sort_values('Year')
    
    # Trace 1: Cột Cán cân thương mại
    fig.add_trace(go.Bar(
        x=country_df['Year'],
        y=country_df['Trade_Balance_USD'],
        name=f'Cán cân thương mại',
        marker_color=np.where(country_df['Trade_Balance_USD'] < 0, 'red', 'green'),
        visible=(country == 'VNM') # Mặc định chỉ hiển thị VNM ban đầu
    ))

    # Trace 2: Đường Độ mở kinh tế
    fig.add_trace(go.Scatter(
        x=country_df['Year'],
        y=country_df['Economic_Openness_Pct'],
        name=f'Độ mở kinh tế',
        yaxis='y2',
        line=dict(color='blue', width=3),
        visible=(country == 'VNM') # Mặc định chỉ hiển thị VNM ban đầu
    ))

# 3. Xây dựng logic cho Menu Dropdown
buttons = []
for i, country in enumerate(countries):
    # Mỗi quốc gia có 2 traces (Bar và Scatter), tạo mảng boolean để bật đúng 2 traces đó
    visibility = [False] * (len(countries) * 2)
    visibility[i*2] = True      # Bật Bar chart của quốc gia thứ i
    visibility[i*2 + 1] = True  # Bật Scatter chart của quốc gia thứ i
    
    button = dict(
        label=country,
        method="update",
        args=[
            {"visible": visibility}, # Hành động 1: Cập nhật dữ liệu hiển thị
            {"title": f"{country}: Chuyển đổi mô hình Thương mại (1990 - 2024)"} # Hành động 2: Đổi tiêu đề
        ]
    )
    buttons.append(button)

# 4. Cập nhật Layout với Dropdown và Trục Y kép
# Đặt chỉ mục mặc định của Dropdown vào 'VNM'
default_index = countries.index('VNM') if 'VNM' in countries else 0

fig.update_layout(
    updatemenus=[
        dict(
            active=default_index,
            buttons=buttons,
            x=1.1, # Đẩy menu ra ngoài góc phải
            y=1.15,
            xanchor="right",
            yanchor="top"
        )
    ],
    title='VNM: Chuyển đổi mô hình Thương mại (1990 - 2024)',
    yaxis=dict(title='Cán cân thương mại (USD)'),
    yaxis2=dict(title='Độ mở kinh tế (% GDP)', overlaying='y', side='right'),
    barmode='group',
    height=600,
    margin=dict(r=100) # Mở rộng lề phải để không bị cắt chữ ở trục Y2
)

fig.show()
fig.write_html(f"fig/Mô hình thương mại.html")

In [2]:
# Bieu do tuong quan moi chi so cho tung quoc gia
selected_countries = ['VNM', 'THA', 'PHL', 'SGP', 'CHN', 'IND', 'IRL', 'DEU', 'ZAF', 'USA', 'JPN']
df_filtered = df[df["Country"].isin(selected_countries)].copy()

numeric_cols = df_filtered.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = {"Year"}
corr_cols = [col for col in numeric_cols if col not in exclude_cols]

for country, g in df_filtered.groupby("Country"):
    corr_matrix = g[corr_cols].corr()
    display(corr_matrix)

    heatmap = go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.index,
        colorscale="RdBu",
        zmin=-1,
        zmax=1,
        colorbar=dict(title="Correlation")
    )

    fig_corr = go.Figure(data=heatmap)
    fig_corr.update_layout(
        title=f"Tuong quan cac chi so - {country}",
        height=900,
        margin=dict(l=140, r=50, t=80, b=140)
    )
    fig_corr.write_html(f"fig/correlation_heatmap_{country}.html")

,FDI_Inflows_USD,Remittances_Pct_GDP,Inflation_CPI_Pct,Lending_Interest_Rate_Pct,Exports_USD,Imports_USD,GDP_Growth_Pct,Unemployment_Pct,Population,GNI_USD,GDP_Current_USD,Labor_Force_Total,Trade_Balance_USD,GDP_Per_Capita_USD,GDP_GNI_Gap_Pct,Economic_Openness_Pct,FDI_to_GDP_Pct,Labor_Participation_Rate_Pct
FDI_Inflows_USD,1.000000,0.239532,-0.220663,-0.527413,0.691638,0.695363,-0.268235,0.669110,0.737932,0.619869,0.618907,0.781655,0.572628,0.628125,-0.168481,0.302701,-0.230520,0.159817
Remittances_Pct_GDP,0.239532,1.000000,-0.068434,-0.156218,0.148419,0.147993,0.091250,0.284498,0.240958,0.097947,0.096828,0.323888,0.131130,0.103235,-0.203919,0.335152,0.056871,0.263432
Inflation_CPI_Pct,-0.220663,-0.068434,1.000000,0.777828,-0.337596,-0.333263,0.497597,-0.492942,-0.461410,-0.326751,-0.326733,-0.472076,-0.320933,-0.329153,-0.105162,-0.108030,0.548172,-0.058960
Lending_Interest_Rate_Pct,-0.527413,-0.156218,0.777828,1.000000,-0.696959,-0.695068,0.556886,-0.778597,-0.844181,-0.679837,-0.679346,-0.821996,-0.615052,-0.682791,0.041138,-0.264594,0.606550,0.030276
Exports_USD,0.691638,0.148419,-0.337596,-0.696959,1.000000,0.997526,-0.597145,0.738801,0.915640,0.983912,0.983721,0.807208,0.880864,0.986092,-0.127050,0.061458,-0.675065,-0.301792
Imports_USD,0.695363,0.147993,-0.333263,-0.695068,0.997526,1.000000,-0.604388,0.728286,0.915607,0.986908,0.986661,0.802442,0.845410,0.988928,-0.118702,0.039580,-0.681529,-0.317192
GDP_Growth_Pct,-0.268235,0.091250,0.497597,0.556886,-0.597145,-0.604388,1.000000,-0.501442,-0.524039,-0.657468,-0.658090,-0.340323,-0.467283,-0.655431,-0.169743,0.405613,0.760452,0.548574
Unemployment_Pct,0.669110,0.284498,-0.492942,-0.778597,0.738801,0.728286,-0.501442,1.000000,0.850001,0.670534,0.669896,0.894117,0.709289,0.676903,-0.178603,0.491304,-0.455541,0.178945
Population,0.737932,0.240958,-0.461410,-0.844181,0.915640,0.915607,-0.524039,0.850001,1.000000,0.885881,0.885300,0.948104,0.791525,0.890100,-0.025729,0.287588,-0.528412,-0.122563
GNI_USD,0.619869,0.097947,-0.326751,-0.679837,0.983912,0.986908,-0.657468,0.670534,0.885881,1.000000,0.999992,0.730482,0.830130,0.999878,-0.067585,-0.081939,-0.722943,-0.449631


,FDI_Inflows_USD,Remittances_Pct_GDP,Inflation_CPI_Pct,Lending_Interest_Rate_Pct,Exports_USD,Imports_USD,GDP_Growth_Pct,Unemployment_Pct,Population,GNI_USD,GDP_Current_USD,Labor_Force_Total,Trade_Balance_USD,GDP_Per_Capita_USD,GDP_GNI_Gap_Pct,Economic_Openness_Pct,FDI_to_GDP_Pct,Labor_Participation_Rate_Pct
FDI_Inflows_USD,1.000000,0.388325,-0.202316,NaN,0.392998,0.396775,-0.081726,-0.340435,0.420237,0.341377,0.337115,0.469381,0.323538,0.326711,-0.335993,0.477692,0.904369,0.385702
Remittances_Pct_GDP,0.388325,1.000000,0.050188,NaN,0.931818,0.922328,-0.171073,-0.821394,0.248397,0.914371,0.908441,0.910607,0.852205,0.913139,-0.946967,0.882249,0.036656,0.971790
Inflation_CPI_Pct,-0.202316,0.050188,1.000000,NaN,0.054316,0.123313,0.165762,-0.274564,-0.051451,0.061639,0.056536,0.009731,-0.271137,0.052499,-0.096846,-0.014301,-0.240173,0.026630
Lending_Interest_Rate_Pct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Exports_USD,0.392998,0.931818,0.054316,NaN,1.000000,0.995108,-0.176990,-0.688630,0.432018,0.983042,0.982543,0.945370,0.890145,0.982517,-0.902099,0.958091,0.066189,0.941219
Imports_USD,0.396775,0.922328,0.123313,NaN,0.995108,1.000000,-0.172880,-0.706331,0.443783,0.982995,0.982016,0.940084,0.840775,0.980801,-0.900117,0.945224,0.072519,0.929792
GDP_Growth_Pct,-0.081726,-0.171073,0.165762,NaN,-0.176990,-0.172880,1.000000,0.050063,-0.362702,-0.241460,-0.246728,-0.244476,-0.172509,-0.237343,0.104585,-0.102433,0.006273,-0.144391
Unemployment_Pct,-0.340435,-0.821394,-0.274564,NaN,-0.688630,-0.706331,0.050063,1.000000,-0.148892,-0.688696,-0.677071,-0.749047,-0.515797,-0.677983,0.781212,-0.597739,-0.068776,-0.818151
Population,0.420237,0.248397,-0.051451,NaN,0.432018,0.443783,-0.362702,-0.148892,1.000000,0.445604,0.448814,0.579621,0.320547,0.412640,-0.267317,0.417391,0.288870,0.282425
GNI_USD,0.341377,0.914371,0.061639,NaN,0.983042,0.982995,-0.241460,-0.688696,0.445604,1.000000,0.999715,0.934648,0.853087,0.998761,-0.910062,0.893346,-0.001356,0.922539


,FDI_Inflows_USD,Remittances_Pct_GDP,Inflation_CPI_Pct,Lending_Interest_Rate_Pct,Exports_USD,Imports_USD,GDP_Growth_Pct,Unemployment_Pct,Population,GNI_USD,GDP_Current_USD,Labor_Force_Total,Trade_Balance_USD,GDP_Per_Capita_USD,GDP_GNI_Gap_Pct,Economic_Openness_Pct,FDI_to_GDP_Pct,Labor_Participation_Rate_Pct
FDI_Inflows_USD,1.000000,0.612018,-0.233554,-0.728740,0.824604,0.825584,-0.131100,-0.254878,0.882124,0.832322,0.831223,0.814747,-0.717392,0.845338,0.083768,0.746848,0.789105,0.302446
Remittances_Pct_GDP,0.612018,1.000000,-0.301354,-0.738659,0.596872,0.612096,0.094327,-0.201448,0.749516,0.544987,0.543779,0.755164,-0.622414,0.562567,-0.379996,0.850968,0.733651,0.621758
Inflation_CPI_Pct,-0.233554,-0.301354,1.000000,0.493841,-0.267520,-0.247766,-0.129798,0.194972,-0.423384,-0.325297,-0.324718,-0.477450,0.090110,-0.314931,0.251713,-0.175273,-0.134173,-0.596925
Lending_Interest_Rate_Pct,-0.728740,-0.738659,0.493841,1.000000,-0.686408,-0.691822,-0.149256,0.163698,-0.846591,-0.680153,-0.679022,-0.820070,0.629838,-0.694972,0.186216,-0.761908,-0.586733,-0.538792
Exports_USD,0.824604,0.596872,-0.267520,-0.686408,1.000000,0.997096,0.123705,-0.668818,0.941399,0.988912,0.988905,0.932077,-0.840901,0.991062,0.304996,0.758302,0.452360,0.561671
Imports_USD,0.825584,0.612096,-0.247766,-0.691822,0.997096,1.000000,0.143006,-0.649606,0.938349,0.980264,0.980124,0.925149,-0.879676,0.984743,0.279266,0.784940,0.470359,0.544711
GDP_Growth_Pct,-0.131100,0.094327,-0.129798,-0.149256,0.123705,0.143006,1.000000,-0.235173,0.104726,0.089318,0.088814,0.141140,-0.243725,0.097831,-0.201864,0.203999,-0.109885,0.248756
Unemployment_Pct,-0.254878,-0.201448,0.194972,0.163698,-0.668818,-0.649606,-0.235173,1.000000,-0.487463,-0.669126,-0.670476,-0.581128,0.439697,-0.646056,-0.373477,-0.243327,0.101886,-0.664785
Population,0.882124,0.749516,-0.423384,-0.846591,0.941399,0.938349,0.104726,-0.487463,1.000000,0.941752,0.941072,0.985817,-0.789383,0.949046,0.058162,0.829985,0.617909,0.633599
GNI_USD,0.832322,0.544987,-0.325297,-0.680153,0.988912,0.980264,0.089318,-0.669126,0.941752,1.000000,0.999995,0.934009,-0.790535,0.998924,0.314455,0.688874,0.427558,0.567092


,FDI_Inflows_USD,Remittances_Pct_GDP,Inflation_CPI_Pct,Lending_Interest_Rate_Pct,Exports_USD,Imports_USD,GDP_Growth_Pct,Unemployment_Pct,Population,GNI_USD,GDP_Current_USD,Labor_Force_Total,Trade_Balance_USD,GDP_Per_Capita_USD,GDP_GNI_Gap_Pct,Economic_Openness_Pct,FDI_to_GDP_Pct,Labor_Participation_Rate_Pct
FDI_Inflows_USD,1.000000,-0.180551,-0.547219,NaN,0.008406,0.033763,0.496613,0.126776,0.180957,0.064498,0.036186,0.097445,-0.060065,0.071492,0.157089,0.248716,0.939966,-0.084613
Remittances_Pct_GDP,-0.180551,1.000000,-0.020075,NaN,-0.775455,-0.792337,-0.078133,0.694555,-0.825258,-0.833915,-0.821688,-0.887764,-0.649530,-0.852738,-0.860033,-0.828735,-0.175181,-0.809277
Inflation_CPI_Pct,-0.547219,-0.020075,1.000000,NaN,0.025067,-0.023851,-0.018438,-0.419021,-0.128602,-0.003723,0.024453,-0.010462,0.152851,0.016154,-0.018092,-0.180065,-0.448133,0.271941
Lending_Interest_Rate_Pct,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Exports_USD,0.008406,-0.775455,0.025067,NaN,1.000000,0.989581,0.036858,-0.405130,0.935705,0.968438,0.986115,0.912945,0.923417,0.971131,0.931209,0.883413,-0.070869,0.511344
Imports_USD,0.033763,-0.792337,-0.023851,NaN,0.989581,1.000000,0.005130,-0.411277,0.951056,0.970615,0.982315,0.922236,0.858538,0.971742,0.933469,0.913313,-0.047384,0.509935
GDP_Growth_Pct,0.496613,-0.078133,-0.018438,NaN,0.036858,0.005130,1.000000,-0.186007,-0.075530,-0.053542,-0.018159,-0.115262,0.117589,-0.014505,0.115695,0.109440,0.443437,-0.157695
Unemployment_Pct,0.126776,0.694555,-0.419021,NaN,-0.405130,-0.411277,-0.186007,1.000000,-0.315326,-0.438458,-0.435668,-0.458540,-0.346466,-0.464158,-0.427405,-0.362300,0.097667,-0.722116
Population,0.180957,-0.825258,-0.128602,NaN,0.935705,0.951056,-0.075530,-0.315326,1.000000,0.972382,0.963581,0.972692,0.797136,0.968389,0.927514,0.916294,0.122376,0.571998
GNI_USD,0.064498,-0.833915,-0.003723,NaN,0.968438,0.970615,-0.053542,-0.438458,0.972382,1.000000,0.994917,0.975618,0.861573,0.996731,0.907485,0.864136,-0.007555,0.639953


,FDI_Inflows_USD,Remittances_Pct_GDP,Inflation_CPI_Pct,Lending_Interest_Rate_Pct,Exports_USD,Imports_USD,GDP_Growth_Pct,Unemployment_Pct,Population,GNI_USD,GDP_Current_USD,Labor_Force_Total,Trade_Balance_USD,GDP_Per_Capita_USD,GDP_GNI_Gap_Pct,Economic_Openness_Pct,FDI_to_GDP_Pct,Labor_Participation_Rate_Pct
FDI_Inflows_USD,1.000000,0.730361,0.037006,-0.243631,0.547350,0.546829,-0.371803,-0.332726,0.019968,0.122353,0.055377,0.511830,-0.387452,0.058852,-0.624595,0.593764,0.994552,0.399986
Remittances_Pct_GDP,0.730361,1.000000,0.364349,-0.310478,0.725920,0.765711,-0.124261,-0.511270,-0.224641,0.060706,-0.037065,0.470622,-0.668564,-0.020767,-0.915918,0.871296,0.755314,0.483144
Inflation_CPI_Pct,0.037006,0.364349,1.000000,0.094164,0.013902,0.099640,0.219630,-0.654008,-0.706444,-0.545402,-0.582912,0.029838,-0.337549,-0.557335,-0.269984,0.305045,0.085353,0.350779
Lending_Interest_Rate_Pct,-0.243631,-0.310478,0.094164,1.000000,-0.506584,-0.515013,-0.029134,-0.286635,-0.480619,-0.156782,-0.111941,0.188140,0.392650,-0.074970,0.395932,-0.489696,-0.237431,0.370007
Exports_USD,0.547350,0.725920,0.013902,-0.506584,1.000000,0.979229,-0.143667,-0.138447,0.349647,0.584614,0.500229,0.238870,-0.632127,0.494450,-0.863245,0.910991,0.544338,0.035668
Imports_USD,0.546829,0.765711,0.099640,-0.515013,0.979229,1.000000,-0.139732,-0.165827,0.276230,0.543043,0.455566,0.221238,-0.776107,0.453108,-0.884102,0.929499,0.551991,0.055354
GDP_Growth_Pct,-0.371803,-0.124261,0.219630,-0.029134,-0.143667,-0.139732,1.000000,-0.062118,-0.245735,-0.172615,-0.160272,-0.292077,0.087179,-0.150702,0.144219,-0.101875,-0.353613,-0.121577
Unemployment_Pct,-0.332726,-0.511270,-0.654008,-0.286635,-0.138447,-0.165827,-0.062118,1.000000,0.708286,0.235895,0.274503,-0.299746,0.203142,0.229919,0.321062,-0.290889,-0.342966,-0.565613
Population,0.019968,-0.224641,-0.706444,-0.480619,0.349647,0.276230,-0.245735,0.708286,1.000000,0.596012,0.605059,-0.198453,0.031796,0.555225,0.011300,0.082974,-0.021367,-0.618607
GNI_USD,0.122353,0.060706,-0.545402,-0.156782,0.584614,0.543043,-0.172615,0.235895,0.596012,1.000000,0.994510,-0.046288,-0.257087,0.992968,-0.195239,0.226675,0.075722,-0.308668


...

,FDI_Inflows_USD,Remittances_Pct_GDP,Inflation_CPI_Pct,Lending_Interest_Rate_Pct,Exports_USD,Imports_USD,GDP_Growth_Pct,Unemployment_Pct,Population,GNI_USD,GDP_Current_USD,Labor_Force_Total,Trade_Balance_USD,GDP_Per_Capita_USD,GDP_GNI_Gap_Pct,Economic_Openness_Pct,FDI_to_GDP_Pct,Labor_Participation_Rate_Pct
FDI_Inflows_USD,1.000000,NaN,0.156459,-0.404338,0.952506,0.938162,-0.212494,-0.112617,0.862648,0.952027,0.966932,0.883250,0.970412,0.967922,0.899402,-0.232199,0.817848,0.882942
Remittances_Pct_GDP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Inflation_CPI_Pct,0.156459,NaN,1.000000,0.115094,0.253197,0.252396,0.229102,-0.331293,0.032499,0.177347,0.200204,0.042115,0.246399,0.220560,0.203907,0.222178,-0.037679,0.034231
Lending_Interest_Rate_Pct,-0.404338,NaN,0.115094,1.000000,-0.479287,-0.501242,0.043250,-0.462838,-0.605841,-0.479181,-0.445056,-0.586831,-0.376346,-0.459302,-0.376252,-0.334478,-0.445116,-0.596059
Exports_USD,0.952506,NaN,0.253197,-0.479287,1.000000,0.997924,-0.298648,-0.072023,0.938805,0.990812,0.993256,0.948698,0.968977,0.994820,0.886120,-0.072461,0.716225,0.939382
Imports_USD,0.938162,NaN,0.252396,-0.501242,0.997924,1.000000,-0.301836,-0.064028,0.950397,0.991178,0.987696,0.958454,0.951048,0.989745,0.862463,-0.041914,0.709562,0.949219
GDP_Growth_Pct,-0.212494,NaN,0.229102,0.043250,-0.298648,-0.301836,1.000000,-0.037958,-0.423361,-0.315282,-0.317522,-0.412427,-0.274768,-0.288372,-0.310931,0.172867,-0.001075,-0.389546
Unemployment_Pct,-0.112617,NaN,-0.331293,-0.462838,-0.072023,-0.064028,-0.037958,1.000000,0.080262,-0.097878,-0.111359,0.054220,-0.099896,-0.104298,-0.003266,0.511396,0.080627,0.075465
Population,0.862648,NaN,0.032499,-0.605841,0.938805,0.950397,-0.423361,0.080262,1.000000,0.955895,0.932687,0.998271,0.857717,0.929093,0.766661,-0.012853,0.699118,0.990673
GNI_USD,0.952027,NaN,0.177347,-0.479181,0.990812,0.991178,-0.315282,-0.097878,0.955895,1.000000,0.994599,0.966706,0.950774,0.992994,0.852937,-0.150634,0.720060,0.956431


,FDI_Inflows_USD,Remittances_Pct_GDP,Inflation_CPI_Pct,Lending_Interest_Rate_Pct,Exports_USD,Imports_USD,GDP_Growth_Pct,Unemployment_Pct,Population,GNI_USD,GDP_Current_USD,Labor_Force_Total,Trade_Balance_USD,GDP_Per_Capita_USD,GDP_GNI_Gap_Pct,Economic_Openness_Pct,FDI_to_GDP_Pct,Labor_Participation_Rate_Pct
FDI_Inflows_USD,1.000000,0.371131,-0.130689,-0.509176,0.596054,0.622847,-0.027020,-0.322825,0.567833,0.503507,0.513361,0.580215,0.150043,0.507679,0.614237,0.635062,0.529847,0.513880
Remittances_Pct_GDP,0.371131,1.000000,-0.378757,-0.481582,0.732093,0.727609,-0.404296,-0.328006,0.651685,0.751543,0.750851,0.609685,0.385190,0.743422,0.266249,0.316941,-0.205936,0.349575
Inflation_CPI_Pct,-0.130689,-0.378757,1.000000,0.625316,-0.429314,-0.373911,0.118460,0.167531,-0.536891,-0.425456,-0.430019,-0.484745,-0.509426,-0.414970,-0.353867,-0.309548,0.198278,-0.240833
Lending_Interest_Rate_Pct,-0.509176,-0.481582,0.625316,1.000000,-0.817900,-0.797025,0.249118,0.496908,-0.926807,-0.768323,-0.771411,-0.919886,-0.515574,-0.756781,-0.580298,-0.811132,0.081763,-0.741186
Exports_USD,0.596054,0.732093,-0.429314,-0.817900,1.000000,0.987995,-0.345598,-0.665779,0.933677,0.978564,0.980884,0.909448,0.557740,0.977251,0.528965,0.669186,-0.217073,0.657222
Imports_USD,0.622847,0.727609,-0.373911,-0.797025,0.987995,1.000000,-0.314788,-0.683619,0.905994,0.970386,0.971466,0.890788,0.422819,0.969869,0.479261,0.670019,-0.212859,0.667624
GDP_Growth_Pct,-0.027020,-0.404296,0.118460,0.249118,-0.345598,-0.314788,1.000000,-0.062915,-0.452266,-0.367545,-0.364398,-0.453371,-0.336002,-0.353535,-0.225334,-0.251374,-0.020158,-0.400183
Unemployment_Pct,-0.322825,-0.328006,0.167531,0.496908,-0.665779,-0.683619,-0.062915,1.000000,-0.540387,-0.669271,-0.673484,-0.543588,-0.232540,-0.689750,-0.380068,-0.364071,0.470847,-0.439714
Population,0.567833,0.651685,-0.536891,-0.926807,0.933677,0.905994,-0.452266,-0.540387,1.000000,0.893448,0.896875,0.988666,0.609265,0.885040,0.641503,0.798519,-0.088745,0.784149
GNI_USD,0.503507,0.751543,-0.425456,-0.768323,0.978564,0.970386,-0.367545,-0.669271,0.893448,1.000000,0.999713,0.854414,0.526609,0.999045,0.422491,0.531225,-0.331917,0.562701


,FDI_Inflows_USD,Remittances_Pct_GDP,Inflation_CPI_Pct,Lending_Interest_Rate_Pct,Exports_USD,Imports_USD,GDP_Growth_Pct,Unemployment_Pct,Population,GNI_USD,GDP_Current_USD,Labor_Force_Total,Trade_Balance_USD,GDP_Per_Capita_USD,GDP_GNI_Gap_Pct,Economic_Openness_Pct,FDI_to_GDP_Pct,Labor_Participation_Rate_Pct
FDI_Inflows_USD,1.000000,0.351662,0.028393,-0.384358,0.731768,0.749428,0.176520,-0.368767,0.741019,0.703098,0.700156,0.747573,-0.687801,0.707014,-0.704323,0.684507,0.746108,-0.229572
Remittances_Pct_GDP,0.351662,1.000000,-0.396284,-0.438001,0.187512,0.256027,-0.095696,0.104231,0.348391,0.169771,0.166810,0.403829,-0.416563,0.184500,-0.513035,0.514324,0.472869,0.285863
Inflation_CPI_Pct,0.028393,-0.396284,1.000000,0.395245,0.038292,0.078683,0.229418,-0.340941,-0.103338,0.079614,0.078976,-0.102715,-0.184210,0.082752,0.091850,-0.132778,-0.082527,0.039311
Lending_Interest_Rate_Pct,-0.384358,-0.438001,0.395245,1.000000,-0.608227,-0.604000,0.264499,-0.432157,-0.693769,-0.566239,-0.564618,-0.653888,0.500680,-0.561282,0.524607,-0.575201,0.042313,0.572614
Exports_USD,0.731768,0.187512,0.038292,-0.608227,1.000000,0.987232,-0.083209,-0.180351,0.965283,0.977121,0.976275,0.947549,-0.801328,0.974862,-0.745083,0.720999,0.167614,-0.512369
Imports_USD,0.749428,0.256027,0.078683,-0.604000,0.987232,1.000000,-0.082745,-0.201492,0.973602,0.980554,0.979783,0.967824,-0.886387,0.982370,-0.769694,0.737070,0.198278,-0.426754
GDP_Growth_Pct,0.176520,-0.095696,0.229418,0.264499,-0.083209,-0.082745,1.000000,-0.467437,-0.141224,-0.080369,-0.084420,-0.135017,0.068926,-0.080215,-0.048748,-0.071038,0.306341,0.116815
Unemployment_Pct,-0.368767,0.104231,-0.340941,-0.432157,-0.180351,-0.201492,-0.467437,1.000000,-0.143651,-0.249393,-0.248198,-0.185436,0.232568,-0.254283,0.225258,0.064348,-0.353623,-0.268223
Population,0.741019,0.348391,-0.103338,-0.693769,0.965283,0.973602,-0.141224,-0.143651,1.000000,0.962593,0.961299,0.992730,-0.851038,0.962599,-0.811955,0.731259,0.214796,-0.442924
GNI_USD,0.703098,0.169771,0.079614,-0.566239,0.977121,0.980554,-0.080369,-0.249393,0.962593,1.000000,0.999875,0.952098,-0.842742,0.999361,-0.730728,0.600428,0.124610,-0.455825


,FDI_Inflows_USD,Remittances_Pct_GDP,Inflation_CPI_Pct,Lending_Interest_Rate_Pct,Exports_USD,Imports_USD,GDP_Growth_Pct,Unemployment_Pct,Population,GNI_USD,GDP_Current_USD,Labor_Force_Total,Trade_Balance_USD,GDP_Per_Capita_USD,GDP_GNI_Gap_Pct,Economic_Openness_Pct,FDI_to_GDP_Pct,Labor_Participation_Rate_Pct
FDI_Inflows_USD,1.000000,0.004883,-0.098005,-0.407879,0.967068,0.971763,-0.287106,-0.469450,0.942678,0.978200,0.978878,0.915120,0.735994,0.980451,0.600920,0.856040,-0.154313,0.779922
Remittances_Pct_GDP,0.004883,1.000000,0.198856,-0.007723,0.006711,0.012309,0.067133,-0.181650,0.027976,0.009145,0.008984,0.032023,-0.058127,0.010226,0.034087,0.064053,-0.023441,0.040692
Inflation_CPI_Pct,-0.098005,0.198856,1.000000,0.381937,-0.224971,-0.203796,-0.019693,-0.360379,-0.086341,-0.180245,-0.184799,-0.017458,-0.424204,-0.165620,-0.130671,-0.041736,0.286045,0.133243
Lending_Interest_Rate_Pct,-0.407879,-0.007723,0.381937,1.000000,-0.421692,-0.423904,0.259198,0.027494,-0.482863,-0.422257,-0.425124,-0.475307,-0.319056,-0.428367,-0.289121,-0.535486,0.387570,-0.427517
Exports_USD,0.967068,0.006711,-0.224971,-0.421692,1.000000,0.998794,-0.291040,-0.385234,0.908272,0.991000,0.991455,0.854265,0.829912,0.984934,0.577357,0.830227,-0.282630,0.671321
Imports_USD,0.971763,0.012309,-0.203796,-0.423904,0.998794,1.000000,-0.297730,-0.388882,0.917039,0.990429,0.990905,0.865659,0.801517,0.985417,0.579658,0.846009,-0.272670,0.688230
GDP_Growth_Pct,-0.287106,0.067133,-0.019693,0.259198,-0.291040,-0.297730,1.000000,-0.007620,-0.350919,-0.305998,-0.305867,-0.337464,-0.161537,-0.310384,-0.373279,-0.360007,0.412181,-0.295745
Unemployment_Pct,-0.469450,-0.181650,-0.360379,0.027494,-0.385234,-0.388882,-0.007620,1.000000,-0.479244,-0.468690,-0.465222,-0.557168,-0.272994,-0.489031,-0.341556,-0.287014,0.129193,-0.644562
Population,0.942678,0.027976,-0.086341,-0.482863,0.908272,0.917039,-0.350919,-0.479244,1.000000,0.939986,0.939667,0.987700,0.641730,0.950655,0.465622,0.928734,-0.287332,0.896354
GNI_USD,0.978200,0.009145,-0.180245,-0.422257,0.991000,0.990429,-0.305998,-0.468690,0.939986,1.000000,0.999912,0.902077,0.815343,0.998893,0.577750,0.833908,-0.292573,0.747291


,FDI_Inflows_USD,Remittances_Pct_GDP,Inflation_CPI_Pct,Lending_Interest_Rate_Pct,Exports_USD,Imports_USD,GDP_Growth_Pct,Unemployment_Pct,Population,GNI_USD,GDP_Current_USD,Labor_Force_Total,Trade_Balance_USD,GDP_Per_Capita_USD,GDP_GNI_Gap_Pct,Economic_Openness_Pct,FDI_to_GDP_Pct,Labor_Participation_Rate_Pct
FDI_Inflows_USD,1.000000,0.309534,-0.185373,-0.434074,0.470650,0.385805,0.225371,0.505031,0.426990,0.421993,0.421070,0.381124,0.499394,0.370414,0.018314,0.381422,0.932809,0.191604
Remittances_Pct_GDP,0.309534,1.000000,-0.597704,-0.758782,0.731867,0.746275,0.039497,0.297390,0.684339,0.727289,0.728876,0.677160,-0.141350,0.708001,0.260653,0.783724,0.280428,0.610174
Inflation_CPI_Pct,-0.185373,-0.597704,1.000000,0.680793,-0.481000,-0.485691,-0.318242,-0.205654,-0.555612,-0.535234,-0.532200,-0.546940,0.062929,-0.509066,0.268253,-0.451132,-0.169441,-0.486426
Lending_Interest_Rate_Pct,-0.434074,-0.758782,0.680793,1.000000,-0.847632,-0.833095,-0.048734,-0.534750,-0.813365,-0.871608,-0.870433,-0.778170,-0.032142,-0.829328,0.031407,-0.712433,-0.326160,-0.583257
Exports_USD,0.470650,0.731867,-0.481000,-0.847632,1.000000,0.987211,-0.002976,0.693203,0.903614,0.979161,0.979027,0.884071,0.010563,0.923250,0.096700,0.872879,0.312771,0.687355
Imports_USD,0.385805,0.746275,-0.485691,-0.833095,0.987211,1.000000,0.036990,0.594284,0.863371,0.975264,0.975772,0.852908,-0.148983,0.939718,0.131228,0.887357,0.238007,0.696349
GDP_Growth_Pct,0.225371,0.039497,-0.318242,-0.048734,-0.002976,0.036990,1.000000,-0.288164,-0.152343,-0.003367,-0.002975,-0.107700,-0.250472,0.075594,-0.048213,0.131959,0.261215,0.096836
Unemployment_Pct,0.505031,0.297390,-0.205654,-0.534750,0.693203,0.594284,-0.288164,1.000000,0.827127,0.607002,0.603714,0.804718,0.572172,0.432118,-0.109912,0.546461,0.353089,0.547919
Population,0.426990,0.684339,-0.555612,-0.813365,0.903614,0.863371,-0.152343,0.827127,1.000000,0.854028,0.852406,0.987893,0.189476,0.718014,-0.031333,0.838051,0.307197,0.784688
GNI_USD,0.421993,0.727289,-0.535234,-0.871608,0.979161,0.975264,-0.003367,0.607002,0.854028,1.000000,0.999934,0.826957,-0.043765,0.973847,0.076033,0.789049,0.250856,0.623179


Nhìn chung, EO của hầu hết các quốc gia đều có dấu hiện sụt giảm sau khủng hoảng tài chính năm 2008 và có dấu hiệu phục hồi dần dần.

1. Nhóm Chuyển đổi từ Nhập siêu sang Xuất siêu (VNM, THA)
Các quốc gia này đã thành công trong việc thay đổi bản chất nền kinh tế từ tiêu thụ sang sản xuất xuất khẩu.

Việt Nam (VNM):

Cột mốc: Trước 2012 chủ yếu là nhập siêu.

Khủng hoảng: Chịu ảnh hưởng bởi khủng hoảng 2008 khiến độ mở kinh tế sụt giảm tạm thời.

Hiện tại: Trở thành "ngôi sao" xuất siêu với độ mở kinh tế cực cao (vượt 180% GDP).

Thái Lan (THA):

Cột mốc: Khủng hoảng tài chính Á châu 1997 là bước ngoặt buộc THA chuyển từ nhập siêu sang xuất siêu để phục hồi.

Hiện tại: Độ mở kinh tế cao (~130% GDP) nhưng cán cân thương mại gần đây biến động mạnh, có dấu hiệu nhập siêu trở lại vào năm 2022.

2. Nhóm Xuất siêu Bền vững (DEU, IRL, SGP)
Những quốc gia này duy trì thặng dư thương mại khổng lồ, đóng vai trò là nguồn cung hàng hóa và dịch vụ cho toàn cầu.

Đức (DEU): Duy trì xuất siêu ổn định ở mức cực cao (>200 tỷ USD). Độ mở kinh tế tăng dần cho thấy sự phụ thuộc vào thị trường bên ngoài.

Ireland (IRL): Mô hình tăng trưởng bùng nổ sau 2015. Độ mở kinh tế "khủng" (>240% GDP) biến đây thành trung tâm trung chuyển xuất khẩu của các tập đoàn đa quốc gia.

Singapore (SGP): Xuất siêu bền vững gắn liền với vị thế cảng biển và trung tâm tài chính. Độ mở kinh tế luôn ở mức cao nhất thế giới (>300% GDP). Sau năm 2008, EO của SGP có sự chững lại, không cao bằng những năm đầu 2000.

3. Nhóm "Thị trường Tiêu thụ" (USA, IND, PHL)
Đây là những nền kinh tế lấy nội lực tiêu dùng làm trọng tâm, chấp nhận nhập siêu để phục vụ nhu cầu trong nước.

Hoa Kỳ (USA): Nhập siêu ngày càng sâu, chạm ngưỡng 1.000 tỷ USD. Độ mở kinh tế thấp (~25% GDP) cho thấy sức mạnh nội địa khổng lồ.

Ấn Độ (IND): Nhập siêu triền miên. Khủng hoảng 2008 và 2012 (khủng hoảng nợ công châu Âu) làm chậm đà tăng độ mở kinh tế của quốc gia này.

Philippines (PHL): (Dựa trên xu hướng khu vực và dữ liệu tương đồng) Thường xuyên nhập siêu do phụ thuộc vào hàng hóa nhập khẩu và kiều hối hỗ trợ tiêu dùng.

4. Nhóm Đảo chiều và Biến động (JPN, CHN, ZAF)
Nhật Bản (JPN): Bước ngoặt 2011 (thảm họa kép động đất - sóng thần) đã chấm dứt kỷ nguyên xuất siêu huy hoàng, đẩy Nhật vào tình trạng nhập siêu kéo dài do chi phí năng lượng.

Trung Quốc (CHN): (Dựa trên bối cảnh chung) Là quốc gia xuất siêu lớn nhất thế giới, tuy nhiên đang có xu hướng giảm dần độ mở kinh tế để tập trung vào "Tuần hoàn nội bộ".

Nam Phi (ZAF): (Dựa trên bối cảnh chung) Cán cân thương mại phụ thuộc nặng nề vào giá tài nguyên/khoáng sản thế giới, thường xuyên biến động theo chu kỳ hàng hóa.

In [3]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. ĐỌC DỮ LIỆU
file_path = 'cleaned_data/information/panel_macro_cleaned.csv'
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Không tìm thấy file tại {file_path}.")

# Lấy danh sách quốc gia đối chiếu
available_countries = countries = ['THA', 'PHL', 'SGP', 'CHN', 'IND', 'IRL', 'DEU', 'ZAF', 'USA', 'JPN']
available_countries.sort()

# 2. ĐỊNH NGHĨA CÁC CHỈ SỐ CẦN VẼ (Cập nhật 8 biểu đồ)
indicators = [
    {'col': 'Economic_Openness_Pct', 'title': '1. Độ mở Kinh tế (% GDP)', 'ylabel': 'Tỷ lệ (%)'},
    {'col': 'GDP_Current_USD', 'title': '2. Quy mô GDP (Tỷ USD)', 'ylabel': 'Tỷ USD', 'divide_by': 1e9},
    {'col': 'GDP_Growth_Pct', 'title': '3. Tăng trưởng GDP (%)', 'ylabel': 'Tốc độ (%)'},
    {'col': 'GDP_GNI_Gap_Pct', 'title': '4. GNI & GDP Gap (% GDP)', 'ylabel': 'Gap (%)'},
    {'col': 'Inflation_CPI_Pct', 'title': '5. Lạm phát (CPI %)', 'ylabel': 'Lạm phát (%)'},
    {'col': 'FDI_to_GDP_Pct', 'title': '6. Tỷ trọng FDI mới / GDP (%)', 'ylabel': 'FDI / GDP (%)'},
    {'col': 'Lending_Interest_Rate_Pct', 'title': '7. Lãi suất cho vay (%)', 'ylabel': 'Lãi suất (%)'},
    {'col': 'Remittances_Pct_GDP', 'title': '8. Kiều hối / GDP (%)', 'ylabel': 'Tỷ lệ (%)'}
]

# 3. HÀM VẼ DASHBOARD TƯƠNG TÁC
def update_plotly_dashboard(selected_country):
    df_plot = df[df['Country'].isin(['VNM', selected_country])].copy()
    
    # Tạo khung 8 biểu đồ (4 hàng x 2 cột)
    fig = make_subplots(
        rows=4, cols=2, 
        subplot_titles=[ind['title'] for ind in indicators],
        vertical_spacing=0.08, horizontal_spacing=0.1
    )
    
    # Tọa độ 8 biểu đồ trên lưới
    coords = [(1,1), (1,2), (2,1), (2,2), (3,1), (3,2), (4,1), (4,2)]
    
    colors = {'VNM': '#d62728', selected_country: '#1f77b4'}
    
    for i, ind in enumerate(indicators):
        row, col_idx = coords[i]
        col_name = ind['col']
        
        plot_data = df_plot.copy()
        if 'divide_by' in ind:
            plot_data[col_name] = plot_data[col_name] / ind['divide_by']
            
        # Vẽ đường cho từng quốc gia
        for country in ['VNM', selected_country]:
            country_data = plot_data[plot_data['Country'] == country]
            
            # Cài đặt độ nét: VNM nét liền đậm, quốc gia kia nét thường
            line_width = 3.5 if country == 'VNM' else 2.0
            
            fig.add_trace(
                go.Scatter(
                    x=country_data['Year'], 
                    y=country_data[col_name],
                    mode='lines+markers',
                    name=country,
                    line=dict(color=colors[country], width=line_width),
                    hovertemplate=f"<b>{country}</b><br>Năm: %{{x}}<br>Giá trị: %{{y:,.2f}}<extra></extra>",
                    showlegend=(i==0) # Chỉ hiển thị chú thích 1 lần
                ),
                row=row, col=col_idx
            )
            
        # Thêm đường mốc 0 (Baseline)
        fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5, row=row, col=col_idx)
        
        # Cập nhật trục X và Y
        fig.update_yaxes(title_text=ind['ylabel'], tickformat=",", row=row, col=col_idx)
        fig.update_xaxes(title_text="Năm", range=[1990, 2024], dtick=5, row=row, col=col_idx)

    # Cấu hình giao diện tổng thể
    fig.update_layout(
        height=1800, 
        # width=None, # Để None để tự dãn theo khung Notebook
        autosize=True, # Tự động điều chỉnh kích thước
        
        # TỐI ƯU KHOẢNG TRẮNG: Giảm bớt lề (margin)
        margin=dict(l=50, r=20, t=100, b=50), 
        
        title_text=f"SO SÁNH KINH TẾ VĨ MÔ: VIỆT NAM VÀ {selected_country}",
        title_font=dict(size=24, color='black'),
        title_x=0.05, # Căn tiêu đề lệch trái một chút để cân bằng
        
        hovermode="x unified",
        template="plotly_white",
        
        # Đưa Legend lên trên và dàn ngang để không chiếm diện tích bên phải
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    # ÉP BIỂU ĐỒ DÃN HẾT CHIỀU RỘNG TRÌNH DUYỆT
    fig.show(config={'responsive': True})
    fig.write_html(f"fig/So_sanh_VNM_va_{selected_country}.html")
    
# 4. TẠO MENU THẢ XUỐNG
dropdown = widgets.Dropdown(
    options=available_countries,
    value='THA', 
    description='Chọn Quốc gia:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Gắn widget với hàm cập nhật
out = widgets.interactive_output(update_plotly_dashboard, {'selected_country': dropdown})

# Hiển thị
display(dropdown, out)


Dropdown(description='Chọn Quốc gia:', index=7, layout=Layout(width='300px'), options=('CHN', 'DEU', 'IND', 'I…

Output()